In [1]:
import pandas as pd

# Load the complaints.csv file into a DataFrame
df = pd.read_csv('/content/complaints.csv')


,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Submitted via,Date sent to company,Company response to consumer,Timely response?,Complaint ID
0,2023-07-24T15:59:41.000Z,"Credit reporting, credit repair services, or o...",Credit repair services,Fraud or scam,NaN,I signed up for bright credit builder account ...,NaN,Bright Capital Inc,PA,19026,NaN,Web,2023-08-31T13:55:17.000Z,Closed with explanation,Yes,7293129.0
1,2024-03-13T22:02:26.000Z,Checking or savings account,Checking account,Managing an account,Problem accessing account,CHIME locked my account and denied me access t...,NaN,Chime Financial Inc,CA,95827,NaN,Web,2024-03-13T22:46:16.000Z,Closed with explanation,Yes,8538232.0
2,2024-08-19T13:57:03.000Z,Mortgage,Conventional home mortgage,Struggling to pay mortgage,Foreclosure,NaN,Company believes it acted appropriately as aut...,"Shellpoint Partners, LLC",NY,11411,NaN,Phone,2024-08-19T15:01:52.000Z,Closed with explanation,Yes,9853285.0
3,2024-08-27T20:24:10.000Z,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Information belongs to someone else,my brother used my information a few years bac...,Company has responded to the consumer and the ...,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",CA,90706,NaN,Web,2024-08-27T20:24:20.000Z,Closed with non-monetary relief,Yes,9935072.0
4,2024-10-04T17:49:27.000Z,Mortgage,FHA mortgage,Trouble during payment process,Trying to communicate with the company to fix ...,I paid off the purchase of my home & learned t...,NaN,"CITIZENS FINANCIAL GROUP, INC.",PA,19128,Older American,Web,2024-10-04T18:16:58.000Z,Closed with explanation,Yes,10346415.0


In [2]:
import re
import string

# Fill NaN values in 'Consumer complaint narrative' with an empty string
df['Consumer complaint narrative'] = df['Consumer complaint narrative'].fillna('')

# Function to clean text
def clean_text(text):
    text = text.lower()
    text = re.sub(r'x+', '', text)           # removes xxxx, xxxxxxxx, any x repetition
    text = re.sub(r'\{.*?\}', '', text)
    text = re.sub(f'[{re.escape(string.punctuation)}]', '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    text = re.sub(r'<.*?>', '', text)
    return text

# Apply the cleaning function to the narrative column
df['clean_narrative'] = df['Consumer complaint narrative'].apply(clean_text)

# Display the original and cleaned narratives for comparison
display(df[['Consumer complaint narrative', 'clean_narrative']].head())

,Consumer complaint narrative,clean_narrative
0,I signed up for bright credit builder account ...,i signed up for bright credit builder account ...
1,CHIME locked my account and denied me access t...,chime locked my account and denied me access t...
2,,
3,my brother used my information a few years bac...,my brother used my information a few years bac...
4,I paid off the purchase of my home & learned t...,i paid off the purchase of my home learned tha...


In [3]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

# Download necessary NLTK data
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

domain_stopwords = {
    # previous ones +
    'account', 'company', 'told', 'said', 'would', 'also', 'get',
    'one', 'us', 'call', 'called', 'time', 'know', 'could', 'never',
    'even', 'still', 'back', 'say', 'make', 'received', 'receive',
    'sent', 'send', 'month', 'year', 'day', 'date', 'number',
    'consumer', 'complaint', 'information', 'please', 'name',
    # add these new ones
    'credit', 'report', 'reporting', 'section', 'usc', 'act',
    'right', 'written', 'pursuant', 'letter', 'agency', 'bureau',
    'fcra', 'furnish', 'person', 'state', 'well', 'made', 'following'
}
stop_words = set(stopwords.words('english')).union(domain_stopwords)
lemmatizer = WordNetLemmatizer()

# Function for tokenization, stopword removal, and lemmatization
def preprocess_text(text):
    tokens = word_tokenize(text)
    filtered_tokens = [word for word in tokens if word not in stop_words]
    lemmas = [lemmatizer.lemmatize(word) for word in filtered_tokens]
    return ' '.join(lemmas)

# Apply preprocessing to the clean_narrative column
df['processed_narrative'] = df['clean_narrative'].apply(preprocess_text)

# Display the original, cleaned, and processed narratives for comparison
display(df[['Consumer complaint narrative', 'clean_narrative', 'processed_narrative']].head())

,Consumer complaint narrative,clean_narrative,processed_narrative
0,I signed up for bright credit builder account ...,i signed up for bright credit builder account ...,signed bright builder paid annual membership f...
1,CHIME locked my account and denied me access t...,chime locked my account and denied me access t...,chime locked denied access money phone wallet ...
2,,,
3,my brother used my information a few years bac...,my brother used my information a few years bac...,brother used year loan found got removed dont
4,I paid off the purchase of my home & learned t...,i paid off the purchase of my home learned tha...,paid purchase home learned loan taken done fra...


In [10]:
initial_rows = len(df)
df = df[df['clean_narrative'].str.strip() != ''].copy()
print(f"Removed {initial_rows - len(df)} blank rows. New DataFrame size: {len(df)}")
display(df[['Consumer complaint narrative', 'clean_narrative', 'processed_narrative']].head())

Removed 119249 blank rows. New DataFrame size: 70692


In [12]:
display(df[['Consumer complaint narrative', 'clean_narrative', 'processed_narrative']].head())

,Consumer complaint narrative,clean_narrative,processed_narrative
0,I signed up for bright credit builder account ...,i signed up for bright credit builder account ...,signed bright builder paid annual membership f...
1,CHIME locked my account and denied me access t...,chime locked my account and denied me access t...,chime locked denied access money phone wallet ...
3,my brother used my information a few years bac...,my brother used my information a few years bac...,brother used year loan found got removed dont
4,I paid off the purchase of my home & learned t...,i paid off the purchase of my home learned tha...,paid purchase home learned loan taken done fra...
6,In XX/XX/year> XXXX I have sent the 3 credit b...,in year i have sent the credit bureau an ident...,identity theft account deleted year put credit...


In [9]:
print(df['Issue'].value_counts().head(30))

Issue
Problem with a purchase shown on your statement                                     1856
Getting a credit card                                                               1101
Other features, terms, or problems                                                   801
Fees or interest                                                                     641
Problem with a credit reporting company's investigation into an existing problem     598
Problem when making payments                                                         553
Closing your account                                                                 443
Advertising and marketing, including promotional offers                              281
Trouble using your card                                                              279
Incorrect information on your report                                                 250
Problem with a purchase or transfer                                                  231
Trouble using t

In [11]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Filter out empty strings from processed_narrative to avoid errors in vectorizers
# and get the indices of non-empty narratives
non_empty_narrative_indices = df[df['processed_narrative'].astype(bool)].index
processed_narratives_non_empty = df.loc[non_empty_narrative_indices, 'processed_narrative']

# Bag-of-Words (BoW) Vectorizer
# Using min_df to ignore terms that appear in too few documents
# and max_df to ignore terms that appear in too many documents
# Max features limits the number of features (words) to avoid too sparse a matrix
bow_vectorizer = CountVectorizer(max_df=0.95, min_df=2, max_features=1000)
bow_matrix = bow_vectorizer.fit_transform(processed_narratives_non_empty)

# TF-IDF Vectorizer
tfidf_vectorizer = TfidfVectorizer(max_df=0.95, min_df=2, max_features=1000)
tfidf_matrix = tfidf_vectorizer.fit_transform(processed_narratives_non_empty)

print(f"Shape of BoW Matrix: {bow_matrix.shape}")
print(f"Shape of TF-IDF Matrix: {tfidf_matrix.shape}")


Shape of BoW Matrix: (70679, 1000)
Shape of TF-IDF Matrix: (70679, 1000)


In [5]:
!pip install gensim
from sklearn.decomposition import LatentDirichletAllocation
from gensim.models import CoherenceModel
from gensim.corpora.dictionary import Dictionary

# Define no_top_words early as it's used in topic extraction
no_top_words = 10

# Get feature names BEFORE the loop, as they are needed inside
bow_feature_names = bow_vectorizer.get_feature_names_out()
tfidf_feature_names = tfidf_vectorizer.get_feature_names_out()

# Prepare the data for Gensim's CoherenceModel (if not already done)
# This assumes processed_narratives_non_empty is available from previous steps
tokenized_narratives = [text.split() for text in processed_narratives_non_empty]
dictionary = Dictionary(tokenized_narratives)

# Define the number of topics and calculate coherence scores
print("Calculating coherence scores for different k values (BoW only):")
coherence_scores_bow = {}
for k in range(4, 7):
    lda_bow = LatentDirichletAllocation(n_components=k, random_state=42)
    lda_bow.fit(bow_matrix)

    bow_topics_for_gensim = []
    for topic in lda_bow.components_:
        top_words = [bow_feature_names[i]
                     for i in topic.argsort()[:-no_top_words - 1:-1]] # Adjusted to use no_top_words
        bow_topics_for_gensim.append(top_words)

    cm = CoherenceModel(
        topics=bow_topics_for_gensim,
        texts=tokenized_narratives,
        dictionary=dictionary,
        coherence='c_v'
    )
    score = cm.get_coherence()
    coherence_scores_bow[k] = score
    print(f"k={k}  |  coherence={score:.4f}")

# Find the best k based on coherence scores
best_k = max(coherence_scores_bow, key=coherence_scores_bow.get)
print(f"\nOptimal k (based on BoW coherence): {best_k} with score {coherence_scores_bow[best_k]:.4f}")

# Re-train LDA models with the optimal k (or a fixed k if preferred)
# Using the best_k found from the loop for final models
k = best_k # Using the optimal k found

# LDA on BoW Matrix
lda_bow = LatentDirichletAllocation(n_components=k, random_state=42)
lda_bow.fit(bow_matrix)

# LDA on TF-IDF Matrix
lda_tfidf = LatentDirichletAllocation(n_components=k, random_state=42)
lda_tfidf.fit(tfidf_matrix)

print(f"\nLDA models re-fitted with {k} components.")

# Function to display topics and their top words
def display_topics(model, feature_names, num_top_words):
    for topic_idx, topic in enumerate(model.components_):
        print(f"Topic {topic_idx + 1}:")
        print(" ".join([feature_names[i]
                        for i in topic.argsort()[:-num_top_words - 1:-1]]))
    print("\n")

print("Top words for each topic (BoW):")
display_topics(lda_bow, bow_feature_names, no_top_words)

print("Top words for each topic (TF-IDF):")
display_topics(lda_tfidf, tfidf_feature_names, no_top_words)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 68.0 MB/s eta 0:00:00
Calculating coherence scores for different k values (BoW only):
k=4  |  coherence=0.5615
k=5  |  coherence=0.4992
k=6  |  coherence=0.5380

Optimal k (based on BoW coherence): 4 with score 0.5615

LDA models re-fitted with 4 components.
Top words for each topic (BoW):
Topic 1:
payment bank card loan debt amount paid money charge due
Topic 2:
consent personal code without authorization creditor account use fraud report
Topic 3:
state balance item privacy right theft identity violated record fair
Topic 4:
account inaccurate inquiry item remove bureau law violation reported dispute


Top words for each topic (TF-IDF):
Topic 1:
payment debt bank card loan collection paid day late amount
Topic 2:
consent report personal code unauthorized authorization knowledge social ftc security
Topic 3:
state privacy right violated instruction fair accordance item without balance
Topic 4:
account item inquiry bureau inaccurate

In [6]:
k = 4
lda_final = LatentDirichletAllocation(n_components=k, random_state=42)
lda_final.fit(bow_matrix)

print(f"Final model: k={k}, Coherence={0.5770:.4f}\n")
print("Top 15 words per topic:")
print("-" * 50)

for topic_idx, topic in enumerate(lda_final.components_):
    top_words = [bow_feature_names[i]
                 for i in topic.argsort()[:-16:-1]]
    print(f"\nTopic {topic_idx + 1}:")
    print("  " + " | ".join(top_words))

Final model: k=4, Coherence=0.5770

Top 15 words per topic:
--------------------------------------------------

Topic 1:
  payment | bank | card | loan | debt | amount | paid | money | charge | due | pay | phone | email | day | check

Topic 2:
  consent | personal | code | without | authorization | creditor | account | use | fraud | report | purpose | open | security | third | social

Topic 3:
  state | balance | item | privacy | right | theft | identity | violated | record | fair | instruction | without | accordance | creditor | may

Topic 4:
  account | inaccurate | inquiry | item | remove | bureau | law | violation | reported | dispute | file | late | request | debt | immediately


In [8]:
# ── Topic naming & assignment ─────────────────────────────────────────────

topic_labels = {
    0: "Billing & Payment Disputes",
    1: "Unauthorized Access & Consent Violations",
    2: "Identity Theft & Privacy Violations",
    3: "Credit Reporting Inaccuracies & FCRA Disputes"
}

final_coherence = coherence_scores_bow[4]   # pulls the actual k=4 score from your loop, no hardcoding

print(f"Final model: k=4, Coherence={final_coherence:.4f}\n")
print("Labeled topics:")
print("-" * 60)

for topic_idx, topic in enumerate(lda_final.components_):
    top_words = [bow_feature_names[i]
                 for i in topic.argsort()[:-16:-1]]
    label = topic_labels[topic_idx]
    print(f"\nTopic {topic_idx + 1}: {label}")
    print("  " + " | ".join(top_words))

# Assign dominant topic to each document
# lda_final was trained on bow_matrix, which was built from processed_narratives_non_empty
# so the row order matches non_empty_narrative_indices exactly
doc_topic_matrix = lda_final.transform(bow_matrix)

df_topics = df.loc[non_empty_narrative_indices].copy()
df_topics['dominant_topic'] = doc_topic_matrix.argmax(axis=1)
df_topics['topic_label'] = df_topics['dominant_topic'].map(topic_labels)

print("\n" + "=" * 60)
print("Document distribution across topics:")
print(df_topics['topic_label'].value_counts())

Final model: k=4, Coherence=0.5615

Labeled topics:
------------------------------------------------------------

Topic 1: Billing & Payment Disputes
  payment | bank | card | loan | debt | amount | paid | money | charge | due | pay | phone | email | day | check

Topic 2: Unauthorized Access & Consent Violations
  consent | personal | code | without | authorization | creditor | account | use | fraud | report | purpose | open | security | third | social

Topic 3: Identity Theft & Privacy Violations
  state | balance | item | privacy | right | theft | identity | violated | record | fair | instruction | without | accordance | creditor | may

Topic 4: Credit Reporting Inaccuracies & FCRA Disputes
  account | inaccurate | inquiry | item | remove | bureau | law | violation | reported | dispute | file | late | request | debt | immediately

Document distribution across topics:
topic_label
Credit Reporting Inaccuracies & FCRA Disputes    28688
Billing & Payment Disputes                       24

In [9]:
# UDAAP Risk Flagging Layer ────────────────────────────────────────

udaap_keywords = [
    'unfair', 'deceptive', 'abusive', 'misleading', 'unauthorized',
    'without consent', 'without notice', 'hidden', 'undisclosed',
    'misrepresent', 'false', 'harass', 'threaten', 'coerce',
    'discriminat', 'exploit', 'predatory', 'bait', 'switch'
]

def flag_udaap_risk(text):
    text_lower = str(text).lower()
    matched = [kw for kw in udaap_keywords if kw in text_lower]
    return len(matched) > 0, matched, len(matched)

flags = df_topics['Consumer complaint narrative'].apply(flag_udaap_risk)
df_topics['udaap_flagged'] = flags.apply(lambda x: x[0])
df_topics['udaap_keywords_matched'] = flags.apply(lambda x: x[1])
df_topics['udaap_risk_count'] = flags.apply(lambda x: x[2])

total = len(df_topics)
flagged = df_topics['udaap_flagged'].sum()

print(f"Total complaints analysed : {total:,}")
print(f"UDAAP-risk flagged        : {flagged:,}  ({100*flagged/total:.1f}%)")
print(f"High risk (2+ keywords)   : {(df_topics['udaap_risk_count'] >= 2).sum():,}")

print("\nUDAAP flag rate by topic:")
print(df_topics.groupby('topic_label')['udaap_flagged'].mean().sort_values(ascending=False))

print("\nSample flagged complaints:")
sample = df_topics[df_topics['udaap_flagged']].head(3)
for _, row in sample.iterrows():
    print(f"\n  Topic   : {row['topic_label']}")
    print(f"  Keywords: {row['udaap_keywords_matched']}")
    print(f"  Excerpt : {str(row['Consumer complaint narrative'])[:200]}...")

Total complaints analysed : 70,679
UDAAP-risk flagged        : 12,255  (17.3%)
High risk (2+ keywords)   : 3,316

UDAAP flag rate by topic:
topic_label
Unauthorized Access & Consent Violations         0.605793
Billing & Payment Disputes                       0.179309
Credit Reporting Inaccuracies & FCRA Disputes    0.162507
Identity Theft & Privacy Violations              0.024467
Name: udaap_flagged, dtype: float64

Sample flagged complaints:

  Topic   : Billing & Payment Disputes
  Keywords: ['harass']
  Excerpt : Source Receivables management Is still engaging me concerning a XXXX and now defunct XXXX bill where a XXXX  manager did a fraudulent transaction on a XXXX  XXXX stealing the phone and not paying XXXX...

  Topic   : Credit Reporting Inaccuracies & FCRA Disputes
  Keywords: ['deceptive', 'false']
  Excerpt : XX/XX/XXXX XXXX XXXX XXXX XXXX,, Experian heretofore has stolen my identity and According to the Fair Credit Reporting Act ( 15 USC 1681 ) and the Gramm-Leach-Bliley A